# EPyMARL Multi-Agent Coverage Training
## Colab/Kaggle Notebook

**Features:**
- ✅ Automated EPyMARL installation
- ✅ Google Drive checkpointing
- ✅ Resume training after timeouts
- ✅ TensorBoard integration
- ✅ 4-5 hour training sessions

**Expected Performance:**
- Coverage: 85-95%
- Training time: 6-12 hours (2-3 Colab sessions)
- GPU recommended (but CPU works)

## 1️⃣ Setup: Mount Google Drive (Colab Only)

In [ ]:
import os
import sys

# Check if running on Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running on Kaggle or local")

# Mount Google Drive (Colab only)
if IN_COLAB:
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/EPyMARL_Coverage'
else:
    # Kaggle: use /kaggle/working
    SAVE_DIR = '/kaggle/working/EPyMARL_Coverage'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {SAVE_DIR}")

## 2️⃣ Install Dependencies

This cell installs:
- Core dependencies (numpy, torch, networkx, pyyaml)
- EPyMARL dependencies (gym, smaclite, protobuf)
- Training tools (tensorboard, sacred)

In [ ]:
%%bash
# Install system dependencies
apt-get update -qq
apt-get install -y -qq git wget > /dev/null 2>&1

# Install Python dependencies for coverage environment
pip install -q numpy networkx torch pyyaml tensorboard sacred

# Install EPyMARL-specific dependencies
pip install -q gym==0.21.0 gymnasium
pip install -q protobuf==3.20.3
pip install -q smaclite

echo "✓ Dependencies installed"

## 3️⃣ Clone and Setup EPyMARL + Coverage Environment

In [ ]:
%%bash
# Clone EPyMARL if not already present
if [ ! -d "epymarl" ]; then
    echo "Cloning EPyMARL..."
    git clone -q https://github.com/uoe-agents/epymarl.git
    cd epymarl
    pip install -q -e .
    cd ..
else
    echo "EPyMARL already cloned"
fi

# Clone coverage environment
if [ ! -d "ind-q" ]; then
    echo "Cloning coverage environment..."
    git clone -q https://github.com/ayyan-k98/ind-q.git
else
    echo "Coverage environment already cloned"
    cd ind-q && git pull -q && cd ..
fi

echo "✓ Repositories ready"

## 4️⃣ Install Coverage Environment to EPyMARL

In [ ]:
# Run automated setup
!python ind-q/epymarl_integration/setup_coverage.py ./epymarl

## 5️⃣ Verify Installation

In [ ]:
# Test environment import
import sys
sys.path.insert(0, './epymarl/src')

try:
    from envs.coverage import CoverageEnv
    
    env = CoverageEnv()
    info = env.get_env_info()
    
    print("✓ Coverage environment installed successfully!")
    print(f"\nEnvironment info:")
    print(f"  Agents: {info['n_agents']}")
    print(f"  Actions: {info['n_actions']}")
    print(f"  Observation size: {info['obs_shape']}")
    print(f"  State size: {info['state_shape']}")
    print(f"  Episode limit: {info['episode_limit']}")
    
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("\nTroubleshooting:")
    print("1. Re-run cell 2 (Install Dependencies)")
    print("2. Re-run cell 3 (Clone repositories)")
    print("3. Re-run cell 4 (Install coverage environment)")
    print("\nIf error persists, restart runtime and run all cells again.")
    raise

## 6️⃣ Create Training Script with Checkpointing

In [ ]:
%%writefile epymarl/train_with_checkpoint.py
#!/usr/bin/env python
import os
import sys
import json
import shutil
from pathlib import Path
import subprocess

class CheckpointManager:
    """Manages checkpointing to Google Drive or Kaggle storage."""
    
    def __init__(self, save_dir):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.metadata_file = self.save_dir / 'training_metadata.json'
        
    def get_latest_checkpoint(self):
        """Find the latest checkpoint."""
        if not self.metadata_file.exists():
            return None
            
        with open(self.metadata_file, 'r') as f:
            metadata = json.load(f)
            
        checkpoint_path = self.save_dir / metadata.get('latest_checkpoint', '')
        if checkpoint_path.exists():
            return str(checkpoint_path)
        return None
    
    def save_checkpoint(self, results_dir, t_env):
        """Save checkpoint to Drive/Kaggle storage."""
        checkpoint_name = f'checkpoint_{t_env}'
        checkpoint_path = self.save_dir / checkpoint_name
        
        # Copy models directory
        models_src = Path(results_dir) / 'models'
        models_dst = checkpoint_path / 'models'
        
        if models_src.exists():
            shutil.copytree(models_src, models_dst, dirs_exist_ok=True)
            print(f"✓ Checkpoint saved: {checkpoint_path}")
            
            # Update metadata
            metadata = {
                'latest_checkpoint': checkpoint_name,
                't_env': t_env,
            }
            with open(self.metadata_file, 'w') as f:
                json.dump(metadata, f, indent=2)
                
            return True
        return False

def train_with_resume(save_dir, config='qmix', env_config='coverage', 
                     t_max=2000000, save_interval=100000, **kwargs):
    """Train with automatic checkpointing and resume."""
    
    manager = CheckpointManager(save_dir)
    
    # Check for existing checkpoint
    checkpoint = manager.get_latest_checkpoint()
    
    # Build command
    cmd = [
        'python', 'src/main.py',
        f'--config={config}',
        f'--env-config={env_config}',
        f'--t_max={t_max}',
    ]
    
    # Add resume checkpoint if available
    if checkpoint:
        print(f"Resuming from checkpoint: {checkpoint}")
        cmd.append(f'--checkpoint_path={checkpoint}/models')
    else:
        print("Starting new training run")
    
    # Add additional arguments
    for key, value in kwargs.items():
        cmd.append(f'--{key}={value}')
    
    print(f"\nCommand: {' '.join(cmd)}\n")
    
    # Run training
    try:
        subprocess.run(cmd, check=True)
    except KeyboardInterrupt:
        print("\n\nTraining interrupted by user")
    except Exception as e:
        print(f"\n\nTraining stopped: {e}")
    
    # Save final checkpoint
    print("\nSaving final checkpoint...")
    manager.save_checkpoint('results', t_max)
    print("✓ Training session complete")

if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--save_dir', type=str, required=True)
    parser.add_argument('--t_max', type=int, default=2000000)
    parser.add_argument('--use_cuda', type=str, default='True')
    args = parser.parse_args()
    
    train_with_resume(
        save_dir=args.save_dir,
        t_max=args.t_max,
        use_cuda=args.use_cuda
    )

## 7️⃣ Start TensorBoard (Optional)

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard
%tensorboard --logdir epymarl/results/tb_logs

## 8️⃣ Train Model

**Note:** Training will automatically:
- Resume from last checkpoint if available
- Save checkpoints every 100K timesteps
- Save to Google Drive/Kaggle storage

**Training time:** ~4-5 hours per session, 2-3 sessions total

In [ ]:
import torch

# Check GPU availability
use_cuda = 'True' if torch.cuda.is_available() else 'False'
print(f"Using CUDA: {use_cuda}")
if use_cuda == 'True':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Start training
os.chdir('epymarl')
!python train_with_checkpoint.py \
    --save_dir="{SAVE_DIR}" \
    --t_max=2000000 \
    --use_cuda={use_cuda}

## 9️⃣ Check Training Progress

In [ ]:
import json
from pathlib import Path

# Check metadata
metadata_file = Path(SAVE_DIR) / 'training_metadata.json'
if metadata_file.exists():
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    print("Training Progress:")
    print(f"  Latest checkpoint: {metadata.get('latest_checkpoint', 'None')}")
    print(f"  Timesteps: {metadata.get('t_env', 0):,} / 2,000,000")
    progress = (metadata.get('t_env', 0) / 2000000) * 100
    print(f"  Progress: {progress:.1f}%")
else:
    print("No training metadata found. Start training first.")

## 🔟 Evaluate Trained Model

In [ ]:
# Find latest checkpoint
import json
from pathlib import Path

metadata_file = Path(SAVE_DIR) / 'training_metadata.json'
if metadata_file.exists():
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    checkpoint_name = metadata.get('latest_checkpoint', '')
    checkpoint_path = Path(SAVE_DIR) / checkpoint_name / 'models'
    
    if checkpoint_path.exists():
        print(f"Evaluating checkpoint: {checkpoint_path}")
        
        os.chdir('/content/epymarl')
        !python src/main.py \
            --config=qmix \
            --env-config=coverage \
            --checkpoint_path="{checkpoint_path}" \
            --evaluate \
            --test_nepisode=20 \
            --render
    else:
        print("No checkpoint found. Train the model first.")
else:
    print("No training metadata found.")

## 📊 Download Results (Optional)

In [ ]:
# Compress results for download
!zip -r -q coverage_results.zip epymarl/results/

if IN_COLAB:
    from google.colab import files
    files.download('coverage_results.zip')
else:
    print("Results saved to: coverage_results.zip")

## 💡 Tips

### For Multiple Sessions:
1. **Session 1 (4-5 hours):** Run cell 8, training will auto-save to Drive
2. **Session 2 (4-5 hours):** Just run cell 8 again - it will auto-resume
3. **Session 3 (if needed):** Repeat until you reach 2M timesteps

### Monitor Progress:
- Use TensorBoard (cell 7) to see live training curves
- Check cell 9 to see how many timesteps completed
- Training is complete when coverage reaches 85-95%

### Expected Timeline:
- **500K steps:** ~60-70% coverage
- **1M steps:** ~75-85% coverage  
- **2M steps:** 85-95% coverage (target)

### If Session Times Out:
- Don't worry! Just restart and run cell 8 again
- Training will automatically resume from last checkpoint
- All progress is saved to Google Drive